# Revision experiments: statistics, size control, unseen obfuscation, DAE sensitivity

**How to use this notebook**
1. *Runtime → Change runtime type → T4 GPU.*
2. In the **Settings** cell, keep `REHEARSAL = True` the first time (a short test with 500 files per class that trains small models and checks that every step runs). When it finishes without an error, set `REHEARSAL = False` and *Run all*.
3. The real run reuses the trained models of the main study from Google Drive (`dae_study_results_full/models`), so parts A and B need **no training**. Parts C, D and E train new models. Every part saves to Google Drive and is skipped when it is already finished, so after a disconnect just *Run all* again.
4. You can switch parts off in the Settings cell (`DO_...`). Suggested order: A and B first (about an hour), then C, D, E one at a time.
5. When a part finishes, send me the printed tables or the csv files from the Drive folder `dae_study_results_revision/results`.

Nothing here executes any malware: files are only read as bytes and turned into images.

| Part | What it answers | Trains new models? |
|---|---|---|
| A | Paired bootstrap confidence intervals and equivalence check; size-controlled (matched) evaluation; near-duplicate check | no |
| B | UPX: number of packed files per class, acceptance rate, UPX version | no |
| C | Unseen obfuscation: leave one type out | yes |
| D | Sensitivity of the DAE result to learning rate, width, loss, training length and training mixture | yes |
| E | Fixed byte budget (no resizing, image side independent of file length) | yes (a full smaller run) |

In [ ]:
# ===================== SETTINGS (the only cell you may edit) =====================
REHEARSAL = False      # True = quick test.  False = the real experiment.

TRAPMINE_DIR = "/content/drive/MyDrive/Drive/Drive_IEEE/PAPER/1. ULUSLARARASI_MAKALE/4_ML for MALWARE/DATASETS/TRAPMINE"

#TRAPMINE_DIR = "/content/drive/MyDrive/Drive_Mesut/Drive_IEEE/PAPER/1. ULUSLARARASI_MAKALE/4_ML for MALWARE/DATASETS/TRAPMINE"
BENIGN_ZIP, BENIGN_PASSWORD   = TRAPMINE_DIR + "/ml-data-benign.zip",    "benign"
MALWARE_ZIP, MALWARE_PASSWORD = TRAPMINE_DIR + "/ml-data-malicious.zip", "infected"

OUT_MAIN = "/content/drive/MyDrive/dae_study_results_full"      # finished main run (models, results)
OUT_UPX  = "/content/drive/MyDrive/dae_study_results_upx"       # finished UPX run (valid_UPX.npy, meta.csv)

DO_A_STATS = True     # bootstrap CIs, size-controlled evaluation, near-duplicates (no training)
DO_B_UPX   = True     # UPX counts (no training)
DO_B_UPX_STATS = True # UPX: bootstrap intervals; packs all files again with UPX, about as long as the UPX build (no training)
DO_C_LOO   = False     # leave-one-obfuscation-out (trains)
DO_D_SENS  = False     # DAE sensitivity (trains)
DO_E_FIXED = False     # fixed byte budget (trains)
# =================================================================================

MODE = "rehearsal" if REHEARSAL else "full"
WORK = f"/content/work_rev_{MODE}"                                  # fast local disk
OUT  = f"/content/drive/MyDrive/dae_study_results_revision" + ("_rehearsal" if REHEARSAL else "")
MAX_PER_CLASS = 500 if REHEARSAL else 10000
SEEDS   = "0" if REHEARSAL else "0 1 2"
CLFS    = "cnn1" if REHEARSAL else "cnn1 cnn2"
EPOCHS  = 3 if REHEARSAL else 20
BOOT    = 200 if REHEARSAL else 2000
ALLOW   = "--allow-train" if REHEARSAL else ""      # the rehearsal has no cached models
LOO_HELD = "Encryption Code_Packing" if REHEARSAL else "Code_Packing Control_Flow_Obfuscation Data_Obfuscation Encryption String_Obfuscation"
SENS_VARIANTS = "default width_0.5 mix_clean_0.5" if REHEARSAL else "all"
SENS_SEEDS = "0" if REHEARSAL else "0 1"            # sensitivity uses validation data; 2 seeds are enough
SENS_CLFS  = "cnn1"
FIXED_TECHS = "Code_Packing Encryption" if REHEARSAL else "Code_Packing Control_Flow_Obfuscation Data_Obfuscation Encryption String_Obfuscation"
SIZE = 128
print(f"mode={MODE}  max_per_class={MAX_PER_CLASS}  seeds={SEEDS}  classifiers={CLFS}  epochs={EPOCHS}  bootstrap={BOOT}")

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("GPU found:", gpus if gpus else "NONE - set Runtime > Change runtime type > T4 GPU, then re-run")
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, subprocess
from pathlib import Path

if shutil.which("7z") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "p7zip-full"], check=True)

def extract(zip_path, out_dir, password):
    if (Path(out_dir) / ".done").exists():
        return
    os.makedirs(out_dir, exist_ok=True)
    r = subprocess.run(["7z", "x", zip_path, f"-o{out_dir}", f"-p{password}", "-y"],
                       capture_output=True, text=True)
    if r.returncode not in (0, 1):
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"7z failed on {zip_path}")
    (Path(out_dir) / ".done").write_text("ok")

extract(BENIGN_ZIP,  "/content/trapmine/benign",  BENIGN_PASSWORD)
extract(MALWARE_ZIP, "/content/trapmine/malware", MALWARE_PASSWORD)
count = lambda d: sum(1 for p in Path(d).rglob("*") if p.is_file() and p.name != ".done")
print("benign files :", count("/content/trapmine/benign"))
print("malware files:", count("/content/trapmine/malware"))

## Code (embedded so this notebook is self-contained - no need to open these)

In [ ]:
import os
os.makedirs('/content/dae_study', exist_ok=True)

In [ ]:
%%writefile /content/dae_study/obfuscate.py
"""
Byte-level obfuscations.

The first six are reproduced EXACTLY as coded in the original notebook
(Using_DAE_for_Detecting_Obfuscated_Malware.ipynb, cell 17), not as
described in the manuscript.  Differences between the two matter:

  Encryption               XOR 0xFF on every byte (bit complement)
  Control_Flow_Obfuscation XOR 0xFF on every 4th byte, from offset 0
  Data_Obfuscation         +1 mod 256 on every byte
  String_Obfuscation       XOR 0xFF on one byte every 32, from offset 512
  Code_Packing             zlib.compress of the whole file
  File_Header_Modification keeps bytes [0:128] and OVERWRITES THE REST
                           WITH 0xFF.  This destroys the file body, so it
                           is NOT invertible.  It is kept as a
                           "destructive control": no method can recover
                           the content, so any gain over identity must
                           come from something other than reconstruction.

`invertible=True` means apply/invert round-trip exactly (checked on every
file at build time).  For those techniques the "oracle" condition equals
the clean condition by construction.

UPX is an optional real-world packer, enabled only if `upx` is on PATH.
"""
import os
import shutil
import subprocess
import tempfile
import zlib
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np


def _arr(b: bytes) -> np.ndarray:
    return np.frombuffer(b, dtype=np.uint8).copy()


def code_packing(b):
    return zlib.compress(b)


def code_packing_inv(b):
    return zlib.decompress(b)


def encryption(b):
    return (_arr(b) ^ 0xFF).tobytes()


def header_mod(b):
    return bytes(b[:128]) + b"\xff" * max(len(b) - 128, 0)


def control_flow(b):
    a = _arr(b)
    a[0::4] ^= 0xFF
    return a.tobytes()


def data_obf(b):
    return (_arr(b) + np.uint8(1)).tobytes()          # uint8 wraps mod 256


def data_obf_inv(b):
    return (_arr(b) - np.uint8(1)).tobytes()


def string_obf(b):
    a = _arr(b)
    a[512::32] ^= 0xFF
    return a.tobytes()


def upx_pack(b):
    with tempfile.TemporaryDirectory() as d:
        src, dst = os.path.join(d, "in.bin"), os.path.join(d, "out.bin")
        with open(src, "wb") as f:
            f.write(b)
        r = subprocess.run(["upx", "-q", "-9", "-o", dst, src],
                           capture_output=True, timeout=120)
        if r.returncode != 0 or not os.path.exists(dst):
            raise RuntimeError("upx failed")
        with open(dst, "rb") as f:
            return f.read()


@dataclass(frozen=True)
class Technique:
    name: str
    apply: Callable[[bytes], bytes]
    invert: Optional[Callable[[bytes], bytes]]   # None => not invertible

    @property
    def invertible(self) -> bool:
        return self.invert is not None


_ALL = [
    Technique("Code_Packing", code_packing, code_packing_inv),
    Technique("Control_Flow_Obfuscation", control_flow, control_flow),   # XOR is an involution
    Technique("Data_Obfuscation", data_obf, data_obf_inv),
    Technique("Encryption", encryption, encryption),
    Technique("File_Header_Modification", header_mod, None),
    Technique("String_Obfuscation", string_obf, string_obf),
]
if shutil.which("upx"):
    _ALL.append(Technique("UPX", upx_pack, None))

TECHNIQUES = {t.name: t for t in _ALL}

In [ ]:
%%writefile /content/dae_study/imaging.py
"""
Binary -> RGB image, matching the original pipeline.

Original: three consecutive bytes -> one pixel; square image of side
int(sqrt(n_pixels)) + 1, remaining pixels black; saved as PNG; later read
with Keras load_img(target_size=(128,128)), whose default interpolation is
NEAREST.  Nearest-neighbour means each output pixel is ONE source pixel, so
for a 1 MB file roughly 95% of the bytes never reach the network.  We keep
that as the default for comparability and expose `resample` so an
area-averaging variant can be run as an ablation.
"""
import numpy as np
from PIL import Image

_RESAMPLE = {"nearest": Image.NEAREST, "bilinear": Image.BILINEAR,
             "area": Image.BOX}


def bytes_to_rgb(b: bytes, size: int = 128, resample: str = "nearest") -> np.ndarray:
    a = np.frombuffer(b, dtype=np.uint8)
    n_pix = len(a) // 3
    side = int(np.sqrt(n_pix)) + 1
    canvas = np.zeros((side * side, 3), dtype=np.uint8)
    if n_pix:
        canvas[:n_pix] = a[: n_pix * 3].reshape(-1, 3)
    img = Image.fromarray(canvas.reshape(side, side, 3), "RGB")
    img = img.resize((size, size), _RESAMPLE[resample])
    return np.asarray(img, dtype=np.uint8)


def bytes_to_rgb_fixed(b: bytes, size: int = 128) -> np.ndarray:
    """
    Fixed byte budget (used for the size-control experiment): the first
    size*size*3 bytes become the size x size image directly, with no
    resizing.  Shorter files are padded with zero bytes.  The image side
    therefore no longer depends on the file length, and no resampling is done.
    """
    a = np.frombuffer(b, dtype=np.uint8)[: size * size * 3]
    canvas = np.zeros(size * size * 3, dtype=np.uint8)
    canvas[: len(a)] = a
    return canvas.reshape(size, size, 3)


def byte_entropy(b: bytes) -> float:
    if not b:
        return 0.0
    c = np.bincount(np.frombuffer(b, np.uint8), minlength=256)
    p = c[c > 0] / len(b)
    return float(-(p * np.log2(p)).sum())

In [ ]:
%%writefile /content/dae_study/data.py
"""
Dataset construction.

Order of operations (this order is the point):
  1. hash every file, drop exact duplicates, drop hashes present in both
     classes (label noise);
  2. build ONE image store holding, for every file, the clean image and the
     image of every obfuscated version -- obfuscation is applied to BOTH
     classes, so every sample has a known clean counterpart;
  3. splits are drawn later, per seed, over file indices.  Nothing is fit
     before the split exists, so nothing can leak across it.
"""
import csv
import hashlib
import json
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

from imaging import bytes_to_rgb, bytes_to_rgb_fixed, byte_entropy
from obfuscate import TECHNIQUES


def sha256_file(p, bs=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while chunk := f.read(bs):
            h.update(chunk)
    return h.hexdigest()


def build_manifest(benign_dir, malware_dir, max_per_class=None):
    rows = []
    for d, label in ((benign_dir, 0), (malware_dir, 1)):
        for p in sorted(Path(d).rglob("*")):      # nested folders are fine
            if p.is_file() and p.stat().st_size > 0:
                rows.append({"path": str(p), "label": label,
                             "sha": sha256_file(p)})
    by_sha = {}
    for r in rows:
        by_sha.setdefault(r["sha"], []).append(r)
    keep, stats = [], {"files_seen": len(rows), "within_class_duplicates_dropped": 0,
                       "cross_class_files_dropped": 0}
    for rs in by_sha.values():
        if len({r["label"] for r in rs}) > 1:
            stats["cross_class_files_dropped"] += len(rs)
            continue
        keep.append(rs[0])
        stats["within_class_duplicates_dropped"] += len(rs) - 1
    keep.sort(key=lambda r: (r["label"], r["path"]))
    if max_per_class:
        rng = np.random.default_rng(12345)
        sel = []
        for label in (0, 1):
            cls = [r for r in keep if r["label"] == label]
            take = rng.permutation(len(cls))[:max_per_class]
            sel += [cls[i] for i in sorted(take)]
        keep = sel
    stats["benign"] = sum(r["label"] == 0 for r in keep)
    stats["malware"] = sum(r["label"] == 1 for r in keep)
    return keep, stats


def make_split(labels, seed):
    """0=train 60%, 1=val 20%, 2=test 20%, stratified by class."""
    idx = np.arange(len(labels))
    tr, rest = train_test_split(idx, test_size=0.4, random_state=seed,
                                stratify=labels)
    va, te = train_test_split(rest, test_size=0.5, random_state=seed,
                              stratify=labels[rest])
    split = np.zeros(len(labels), dtype=np.int8)
    split[va], split[te] = 1, 2
    assert not (set(tr) & set(va)) and not (set(tr) & set(te)) \
        and not (set(va) & set(te)), "split overlap"
    return split


def _process(args):
    i, path, size, resample, techs, mode = args
    to_img = (lambda x: bytes_to_rgb_fixed(x, size)) if mode == "fixed" \
        else (lambda x: bytes_to_rgb(x, size, resample))
    b = Path(path).read_bytes()
    out = {"clean": to_img(b)}
    valid = {}
    for name in techs:
        t = TECHNIQUES[name]
        try:
            ob = t.apply(b)
        except Exception:
            out[name] = np.zeros((size, size, 3), np.uint8)
            valid[name] = False
            continue
        if t.invertible and t.invert(ob) != b:
            raise RuntimeError(f"{name} does not round-trip on {path}")
        out[name] = to_img(ob)
        valid[name] = True
    return i, out, valid, len(b), byte_entropy(b)


def build_store(benign_dir, malware_dir, work, size=128, resample="nearest",
                max_per_class=None, workers=2, techniques=None, mode="resize"):
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    if (work / "build_done.flag").exists():
        print("store already built; delete build_done.flag to rebuild")
        return
    techs = techniques or list(TECHNIQUES)
    manifest, stats = build_manifest(benign_dir, malware_dir, max_per_class)
    n = len(manifest)
    print(f"{n} files after dedup: {stats}")
    names = ["clean"] + techs
    mm = {k: np.lib.format.open_memmap(work / f"{k}.npy", mode="w+",
                                       dtype=np.uint8, shape=(n, size, size, 3))
          for k in names}
    valid = {t: np.ones(n, bool) for t in techs}
    nbytes, ent = np.zeros(n, np.int64), np.zeros(n)
    jobs = ((i, r["path"], size, resample, techs, mode) for i, r in enumerate(manifest))
    with ProcessPoolExecutor(max_workers=workers) as ex:
        for k, (i, out, v, nb, e) in enumerate(ex.map(_process, jobs, chunksize=8)):
            for name in names:
                mm[name][i] = out[name]
            for t in techs:
                valid[t][i] = v[t]
            nbytes[i], ent[i] = nb, e
            if (k + 1) % 500 == 0:
                print(f"  {k + 1}/{n}")
    for a in mm.values():
        a.flush()
    for t in techs:
        np.save(work / f"valid_{t}.npy", valid[t])
    with open(work / "meta.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["idx", "label", "sha256", "path", "bytes", "entropy"])
        for i, r in enumerate(manifest):
            w.writerow([i, r["label"], r["sha"], r["path"], nbytes[i], f"{ent[i]:.4f}"])
    np.save(work / "labels.npy", np.array([r["label"] for r in manifest], np.int8))
    (work / "store.json").write_text(json.dumps(
        {"size": size, "resample": resample, "mode": mode, "n": n, "techniques": techs,
         "invertible": [t for t in techs if TECHNIQUES[t].invertible],
         "manifest_stats": stats,
         "valid_fraction": {t: float(valid[t].mean()) for t in techs}}, indent=2))
    (work / "build_done.flag").write_text("ok")
    print("store built")


class Store:
    def __init__(self, work):
        work = Path(work)
        self.work = work
        self.info = json.loads((work / "store.json").read_text())
        self.size = self.info["size"]
        self.techs = self.info["techniques"]
        self.arrays = {k: np.load(work / f"{k}.npy", mmap_mode="r")
                       for k in ["clean"] + self.techs}
        self.valid = {"clean": np.ones(self.info["n"], bool)}
        for t in self.techs:
            self.valid[t] = np.load(work / f"valid_{t}.npy")
        self.labels = np.load(work / "labels.npy")

In [ ]:
%%writefile /content/dae_study/models.py
"""
Models, batching and a manual training loop.

A manual loop (train_on_batch + best-weights early stopping) is used
instead of Sequence/fit so the code behaves the same across TF/Keras
versions.
"""
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


# ------------------------------------------------------------ architectures
def build_dae(size, arch="paper", width=1.0, lr=None, loss="mse"):
    """
    arch="paper": the paper's architecture (max-pool bottleneck, no skips).
    arch="unet" : same widths plus skip connections at full resolution, so it
                  CAN reproduce pixel-level detail.  This is the best case for
                  the DAE: if even this does not help, the conclusion is not an
                  artefact of a weak autoencoder.
    """
    # width, lr and loss are only changed by the sensitivity analysis
    # (extra.py sens); the defaults reproduce the architectures of the paper.
    w = lambda f: max(4, int(round(f * width)))
    if arch == "paper":      # called "baseline DAE" in the manuscript
        m = models.Sequential([
            layers.Input(shape=(size, size, 3)),
            layers.Conv2D(w(32), 3, activation="relu", padding="same"),
            layers.MaxPooling2D(2, padding="same"),
            layers.Conv2D(w(64), 3, activation="relu", padding="same"),
            layers.MaxPooling2D(2, padding="same"),
            layers.Conv2D(w(128), 3, activation="relu", padding="same"),
            layers.UpSampling2D(2),
            layers.Conv2D(w(64), 3, activation="relu", padding="same"),
            layers.UpSampling2D(2),
            layers.Conv2D(w(32), 3, activation="relu", padding="same"),
            layers.Conv2D(3, 3, activation="sigmoid", padding="same"),
        ])
    elif arch == "unet":
        L = layers
        conv = lambda f: L.Conv2D(f, 3, activation="relu", padding="same")
        inp = L.Input(shape=(size, size, 3))
        conv = lambda f: L.Conv2D(w(f), 3, activation="relu", padding="same")
        c1 = conv(32)(conv(32)(inp))
        c2 = conv(64)(conv(64)(L.MaxPooling2D(2)(c1)))
        b = conv(128)(conv(128)(L.MaxPooling2D(2)(c2)))
        d2 = conv(64)(conv(64)(L.Concatenate()([L.UpSampling2D(2)(b), c2])))
        d1 = conv(32)(conv(32)(L.Concatenate()([L.UpSampling2D(2)(d2), c1])))
        m = models.Model(inp, L.Conv2D(3, 1, activation="sigmoid")(d1))
    else:
        raise ValueError(arch)
    opt = "adam" if lr is None else tf.keras.optimizers.Adam(lr)
    m.compile(optimizer=opt, loss=loss)
    return m


class Blur:
    """
    Non-learned low-pass baseline: k x k average pooling, then bilinear
    upsampling back.  Has the same call interface as a Keras model so it can
    be dropped in wherever a DAE is used.  If a DAE does no better than this,
    it is acting as a blur filter, not as a de-obfuscator.
    """
    def __init__(self, k=4):
        self.k = k

    def predict_on_batch(self, x):
        y = tf.nn.avg_pool2d(x, self.k, self.k, "VALID")
        return tf.image.resize(y, x.shape[1:3], method="bilinear").numpy()


def build_cnn(kind, size):
    L = layers
    if kind == "cnn1":      # paper's CNN Model 1: 3 conv + dense64
        stack = [L.Conv2D(32, 3, activation="relu", padding="same"), L.MaxPooling2D(2),
                 L.Conv2D(64, 3, activation="relu", padding="same"), L.MaxPooling2D(2),
                 L.Conv2D(128, 3, activation="relu", padding="same"), L.MaxPooling2D(2),
                 L.Flatten(), L.Dense(64, activation="relu")]
    elif kind == "cnn2":    # deeper, dropout-regularised
        stack = []
        for f in (32, 64, 128):
            stack += [L.Conv2D(f, 3, activation="relu", padding="same"),
                      L.Conv2D(f, 3, activation="relu", padding="same"),
                      L.MaxPooling2D(2), L.Dropout(0.25)]
        stack += [L.Flatten(), L.Dense(128, activation="relu"), L.Dropout(0.5),
                  L.Dense(64, activation="relu")]
    else:
        raise ValueError(kind)
    m = models.Sequential([L.Input(shape=(size, size, 3))] + stack
                          + [L.Dense(1, activation="sigmoid")])
    m.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss="binary_crossentropy")
    return m


# ------------------------------------------------------------------ batching
def batch_iter(store, idx, sources, batch, rng, target=None, dae=None,
               shuffle=True, p_clean=None):
    """
    Yield (x, y).

    sources : list of store keys.  With several, each sample draws one of
              its VALID sources at random (re-drawn every call, i.e. every
              epoch) -- this is how augmentation / the union DAE mix
              clean and obfuscated versions.
    target  : None -> y = class label;  'clean' -> y = clean image (DAE).
    dae     : optional Keras model applied to x (normalisation step).
    p_clean : None (default) -> every valid source is equally likely.
              A number in [0,1] -> the clean image is drawn with that
              probability and the obfuscated forms share the rest (used only
              by the training-mixture sensitivity analysis).
    """
    idx = np.asarray(idx)
    if shuffle:
        idx = rng.permutation(idx)
    for s in range(0, len(idx), batch):
        b = np.sort(idx[s:s + batch])
        if len(sources) == 1:
            src = np.zeros(len(b), int)
        else:
            V = np.stack([store.valid[n][b] for n in sources], 1)
            scores = rng.random(V.shape)
            scores[~V] = -1.0
            if p_clean is None:
                src = scores.argmax(1)
            else:
                assert sources[0] == "clean"
                u = rng.random(len(b))
                scores[:, 0] = -1.0
                src = scores.argmax(1)
                src[(u < p_clean) | (scores.max(1) < 0)] = 0
        x = np.empty((len(b), store.size, store.size, 3), np.float32)
        for k, name in enumerate(sources):
            m = src == k
            if m.any():
                x[m] = store.arrays[name][b[m]] / 255.0
        if dae is not None:
            x = np.clip(dae.predict_on_batch(x), 0.0, 1.0).astype(np.float32)
        if target is None:
            y = store.labels[b].astype(np.float32)
        else:
            y = store.arrays[target][b].astype(np.float32) / 255.0
        yield x, y


def predict_scores(model, store, idx, source, dae=None, batch=128):
    out = []
    for x, _ in batch_iter(store, idx, [source], batch, None, dae=dae,
                           shuffle=False):
        out.append(np.asarray(model.predict_on_batch(x)).reshape(-1))
    return np.concatenate(out) if out else np.zeros(0)


# ------------------------------------------------------------------ training
def _scalar(r):
    return float(np.asarray(r).reshape(-1)[0])


def fit(model, train_batches, val_batches, epochs, patience, tag=""):
    """train_batches(): generator factory; val_batches(): generator factory."""
    best, best_w, wait = np.inf, None, 0
    for ep in range(epochs):
        tl = [_scalar(model.train_on_batch(x, y)) for x, y in train_batches()]
        vl = [_scalar(model.test_on_batch(x, y)) for x, y in val_batches()]
        tl, vl = float(np.mean(tl)), float(np.mean(vl))
        print(f"    {tag} epoch {ep + 1}/{epochs} train={tl:.4f} val={vl:.4f}")
        if vl < best - 1e-5:
            best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_w is not None:
        model.set_weights(best_w)
    return model

In [ ]:
%%writefile /content/dae_study/metrics.py
"""Classification and reconstruction metrics."""
import numpy as np
import tensorflow as tf
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score, roc_curve



def clf_metrics(y, s):
    y = np.asarray(y).astype(int)
    out = {"n_test": int(len(y)), "n_malware": int(y.sum())}
    if len(set(y)) < 2:
        return {**out, **{k: float("nan") for k in
                ("auroc", "auprc", "tpr@fpr1e-2", "tpr@fpr1e-3", "acc", "sens", "spec")}}
    fpr, tpr, _ = roc_curve(y, s)
    pred = (s > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {**out,
            "auroc": float(roc_auc_score(y, s)),
            "auprc": float(average_precision_score(y, s)),
            "tpr@fpr1e-2": float(np.interp(1e-2, fpr, tpr)),
            # only meaningful when n_benign_test >> 1000; see n_test
            "tpr@fpr1e-3": float(np.interp(1e-3, fpr, tpr)),
            "acc": float((tp + tn) / len(y)),
            "sens": float(tp / max(tp + fn, 1)),
            "spec": float(tn / max(tn + fp, 1))}


def _psnr(mse):
    return 10.0 * np.log10(1.0 / np.maximum(mse, 1e-10))


def recon_metrics(store, dae, idx, source, batch=128):
    """
    Per-sample reconstruction quality of dae(source image) against the clean
    image, and of the do-nothing baseline (source image vs clean).
    Returns dict of arrays aligned with sorted(idx) batches.
    """
    mse, ssim, mse0, ssim0, ids = [], [], [], [], []
    idx = np.asarray(idx)
    for s in range(0, len(idx), batch):
        b = np.sort(idx[s:s + batch])
        x = store.arrays[source][b].astype(np.float32) / 255.0
        c = store.arrays["clean"][b].astype(np.float32) / 255.0
        r = np.clip(dae.predict_on_batch(x), 0.0, 1.0).astype(np.float32)
        mse.append(((r - c) ** 2).mean((1, 2, 3)))
        mse0.append(((x - c) ** 2).mean((1, 2, 3)))
        ssim.append(tf.image.ssim(r, c, max_val=1.0).numpy())
        ssim0.append(tf.image.ssim(x, c, max_val=1.0).numpy())
        ids.append(b)
    cat = lambda v: np.concatenate(v) if v else np.zeros(0)
    return {"idx": cat(ids), "mse": cat(mse), "ssim": cat(ssim),
            "mse_identity": cat(mse0), "ssim_identity": cat(ssim0)}


def summarize_recon(rec, labels, which):
    """which: 'benign' | 'malware' | 'all'."""
    lab = labels[rec["idx"]]
    m = np.ones(len(lab), bool) if which == "all" else \
        (lab == (0 if which == "benign" else 1))
    if not m.any():
        return None
    mse, mse0 = rec["mse"][m].mean(), rec["mse_identity"][m].mean()
    return {"cls": which, "n": int(m.sum()),
            "mse": float(mse), "psnr": float(_psnr(mse)),
            "ssim": float(rec["ssim"][m].mean()),
            "mse_identity": float(mse0), "psnr_identity": float(_psnr(mse0)),
            "ssim_identity": float(rec["ssim_identity"][m].mean())}

In [ ]:
%%writefile /content/dae_study/run_experiment.py
"""
Do Denoising Autoencoders Improve Static Malware Detection Under
Obfuscation?  A controlled evaluation.

  python run_experiment.py build  --benign DIR --malware DIR --work OUT
  python run_experiment.py run    --work OUT --seeds 0 1 2 --clfs cnn1 cnn2
  python report.py                --work OUT

Protocol
--------
* Obfuscation is applied to BOTH classes, so every file has a known clean
  version.  Evaluation is on obfuscated TEST files of both classes.
* Whatever preprocessing a condition uses is applied to EVERY test file.
* Splits (60/20/20, stratified) are drawn per seed over deduplicated files;
  DAEs and classifiers are fit on the train split only.
* Everything trained is cached in OUT/models, so runs are resumable and a DAE
  variant can be added later without retraining the baselines.

Conditions (per obfuscation technique T, per classifier)
--------------------------------------------------------
  clean_ref      trained on clean, tested on CLEAN                (reference)
  identity       trained on clean, tested on obfuscated T         (no defence)
  augmentation   trained on clean+obfuscated mix, NO preprocessing
  blur           trained on clean, tested on blur(obf T)          (non-learned)
  blur_matched   trained on blur(clean+obf mix), tested on blur(obf T)
  dae_t          trained on clean, tested on DAE_T(obf T)   (technique known)
  dae_u          trained on clean, tested on DAE_U(obf T)   (one union DAE)
  dae_u_matched  trained on DAE_U(clean+obf mix), tested on DAE_U(obf T)
  oracle         exact analytic inverse (invertible T only; = clean_ref)

DAE architectures: "paper" (the original) and "unet" (skip connections: the
best case for the DAE).  The DAE is the proposed method; blur and
augmentation are the baselines it has to beat.
"""
import argparse
import json
from pathlib import Path

import numpy as np
import tensorflow as tf

from data import Store, build_store, make_split
from metrics import clf_metrics, recon_metrics, summarize_recon
from models import Blur, batch_iter, build_cnn, build_dae, fit, predict_scores


def append(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")


def load(path):
    return tf.keras.models.load_model(path, compile=False)


# ------------------------------------------------------------------- training
def get_dae(a, store, seed, arch, name, sources, dtr, dva):
    path = Path(a.work) / "models" / f"dae_{arch}_{name}_{a.dae_train}_seed{seed}.keras"
    if path.exists():
        return load(path)
    tf.keras.utils.set_random_seed(seed)
    m = build_dae(store.size, arch)
    rng = np.random.default_rng(seed)
    fit(m,
        lambda: batch_iter(store, dtr, sources, a.batch, rng, target="clean"),
        lambda: batch_iter(store, dva, sources, a.batch, np.random.default_rng(999),
                           target="clean", shuffle=False),
        a.dae_epochs, a.patience, tag=f"DAE-{arch}[{name}] seed{seed}")
    m.save(path)
    return m


def get_clf(a, store, kind, seed, cache, tr, va, sources, pre):
    """Train (or load from cache) a classifier; `pre` is applied to its inputs."""
    path = Path(a.work) / "models" / f"clf_{kind}_{cache}_seed{seed}.keras"
    if path.exists():
        return load(path)
    tf.keras.utils.set_random_seed(seed)
    m = build_cnn(kind, store.size)
    rng = np.random.default_rng(seed)
    fit(m,
        lambda: batch_iter(store, tr, sources, a.batch, rng, dae=pre),
        lambda: batch_iter(store, va, sources, a.batch, np.random.default_rng(999),
                           dae=pre, shuffle=False),
        a.clf_epochs, a.patience, tag=f"{kind}[{cache}] seed{seed}")
    m.save(path)
    return m


# ------------------------------------------------------------------- figures
def make_figures(a, store, arch, daes, blur, te):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    labels = store.labels
    for t in store.techs:
        idx = te[store.valid[t][te]]
        ben, mal = idx[labels[idx] == 0][:4], idx[labels[idx] == 1][:4]
        pick = np.concatenate([ben, mal])
        if len(pick) == 0:
            continue
        rows = [("obfuscated", t, None), ("blur baseline", t, blur),
                (f"DAE_T ({arch})", t, daes[t]), (f"DAE_U ({arch})", t, daes["UNION"]),
                ("clean (target)", "clean", None)]
        fig, ax = plt.subplots(len(rows), len(pick),
                               figsize=(1.6 * len(pick), 1.7 * len(rows)), squeeze=False)
        for r, (title, src, model) in enumerate(rows):
            x = store.arrays[src][pick].astype(np.float32) / 255.0
            if model is not None:
                x = np.clip(model.predict_on_batch(x), 0, 1)
            for c in range(len(pick)):
                ax[r, c].imshow(x[c])
                ax[r, c].axis("off")
                if c == 0:
                    ax[r, c].set_title(title, fontsize=8, loc="left")
        for c in range(len(pick)):
            ax[0, c].text(0.5, 1.25, "benign" if labels[pick[c]] == 0 else "malware",
                          transform=ax[0, c].transAxes, ha="center", fontsize=7)
        fig.suptitle(f"{t}", fontsize=9)
        fig.tight_layout()
        fig.savefig(Path(a.work) / "figures" / f"recon_{arch}_{t}.png", dpi=130)
        plt.close(fig)


# ---------------------------------------------------------------------- main
def cmd_run(a):
    work = Path(a.work)
    store = Store(work)
    for d in ("models", "results", "figures"):
        (work / d).mkdir(exist_ok=True)
    res = work / "results"
    raw_base, rec_base = res / "raw_base.jsonl", res / "recon_base.jsonl"
    raw_dae = res / f"raw_{a.dae_train}.jsonl"
    rec_dae = res / f"recon_{a.dae_train}.jsonl"
    labels, techs = store.labels, store.techs
    invertible = set(store.info["invertible"])
    blur = Blur(4)

    for seed in a.seeds:
        split = make_split(labels, seed)
        tr, va, te = (np.where(split == k)[0] for k in (0, 1, 2))
        if a.dae_train == "benign":
            dtr, dva = tr[labels[tr] == 0], va[labels[va] == 0]
        else:
            dtr, dva = tr, va

        # ---- blur reconstruction quality (once per seed)
        flag = res / f"recon_base_seed{seed}.flag"
        if not flag.exists():
            for t in techs:
                te_t = te[store.valid[t][te]]
                r = recon_metrics(store, blur, te_t, t)
                for which in ("benign", "malware", "all"):
                    s = summarize_recon(r, labels, which)
                    if s:
                        append(rec_base, {"seed": seed, "dae_train": "none",
                                          "dae_arch": "none", "dae": "BLUR", "tech": t, **s})
            flag.write_text("ok")

        # ---- DAEs + their reconstruction quality (per architecture)
        daes = {}
        for arch in a.dae_archs:
            d = {t: get_dae(a, store, seed, arch, t, [t], dtr, dva) for t in techs}
            d["UNION"] = get_dae(a, store, seed, arch, "UNION", ["clean"] + techs, dtr, dva)
            daes[arch] = d
            flag = res / f"recon_{a.dae_train}_{arch}_seed{seed}.flag"
            if not flag.exists():
                for t in techs:
                    te_t = te[store.valid[t][te]]
                    for dname, model in (("T", d[t]), ("UNION", d["UNION"])):
                        r = recon_metrics(store, model, te_t, t)
                        for which in ("benign", "malware", "all"):
                            s = summarize_recon(r, labels, which)
                            if s:
                                append(rec_dae, {"seed": seed, "dae_train": a.dae_train,
                                                 "dae_arch": arch, "dae": dname, "tech": t, **s})
                r = recon_metrics(store, d["UNION"], te, "clean")
                for which in ("benign", "malware", "all"):
                    s = summarize_recon(r, labels, which)
                    if s:
                        append(rec_dae, {"seed": seed, "dae_train": a.dae_train,
                                         "dae_arch": arch, "dae": "UNION", "tech": "clean", **s})
                if seed == a.seeds[0]:
                    make_figures(a, store, arch, d, blur, te)
                flag.write_text("ok")

        for kind in a.clfs:
            # ------------------------------------------------ baselines group
            flag = res / f"done_base_seed{seed}_{kind}.flag"
            if not flag.exists():
                clf_clean = get_clf(a, store, kind, seed, "clean", tr, va, ["clean"], None)
                clf_aug = get_clf(a, store, kind, seed, "aug", tr, va, ["clean"] + techs, None)
                clf_bm = get_clf(a, store, kind, seed, "blurmatched", tr, va,
                                 ["clean"] + techs, blur)

                def logb(tech, cond, idx, s):
                    append(raw_base, {"seed": seed, "clf": kind, "tech": tech,
                                      "condition": cond, "dae_train": "none",
                                      "dae_arch": "none", **clf_metrics(labels[idx], s)})

                s_clean = predict_scores(clf_clean, store, te, "clean")
                logb("clean", "clean_ref", te, s_clean)
                logb("clean", "augmentation", te, predict_scores(clf_aug, store, te, "clean"))
                logb("clean", "blur", te, predict_scores(clf_clean, store, te, "clean", dae=blur))
                logb("clean", "blur_matched", te,
                     predict_scores(clf_bm, store, te, "clean", dae=blur))
                for t in techs:
                    mask = store.valid[t][te]
                    idx = te[mask]
                    if len(idx) == 0:
                        continue
                    logb(t, "identity", idx, predict_scores(clf_clean, store, idx, t))
                    logb(t, "augmentation", idx, predict_scores(clf_aug, store, idx, t))
                    logb(t, "blur", idx, predict_scores(clf_clean, store, idx, t, dae=blur))
                    logb(t, "blur_matched", idx, predict_scores(clf_bm, store, idx, t, dae=blur))
                    if t in invertible:
                        logb(t, "oracle", idx, s_clean[mask])
                    else:
                        # clean images of the SAME files, so a defence is compared fairly
                        # when only some files could be transformed (e.g. UPX refuses some)
                        logb(t, "clean_subset", idx, s_clean[mask])
                flag.write_text("ok")
                print(f"seed {seed} / {kind} / baselines complete")

            # ------------------------------------------------- DAE groups
            for arch in a.dae_archs:
                flag = res / f"done_{a.dae_train}_{arch}_seed{seed}_{kind}.flag"
                if flag.exists():
                    continue
                d = daes[arch]
                du = d["UNION"]
                clf_clean = get_clf(a, store, kind, seed, "clean", tr, va, ["clean"], None)
                clf_dm = get_clf(a, store, kind, seed, f"daematched-{arch}-{a.dae_train}",
                                 tr, va, ["clean"] + techs, du)

                def logd(tech, cond, idx, s):
                    append(raw_dae, {"seed": seed, "clf": kind, "tech": tech,
                                     "condition": cond, "dae_train": a.dae_train,
                                     "dae_arch": arch, **clf_metrics(labels[idx], s)})

                logd("clean", "dae_u", te, predict_scores(clf_clean, store, te, "clean", dae=du))
                logd("clean", "dae_u_matched", te,
                     predict_scores(clf_dm, store, te, "clean", dae=du))
                for t in techs:
                    idx = te[store.valid[t][te]]
                    if len(idx) == 0:
                        continue
                    logd(t, "dae_t", idx, predict_scores(clf_clean, store, idx, t, dae=d[t]))
                    logd(t, "dae_u", idx, predict_scores(clf_clean, store, idx, t, dae=du))
                    logd(t, "dae_u_matched", idx, predict_scores(clf_dm, store, idx, t, dae=du))
                flag.write_text("ok")
                print(f"seed {seed} / {kind} / DAE-{arch} complete")


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    b = sub.add_parser("build")
    b.add_argument("--benign", required=True)
    b.add_argument("--malware", required=True)
    b.add_argument("--work", required=True)
    b.add_argument("--size", type=int, default=128)
    b.add_argument("--resample", default="nearest", choices=["nearest", "bilinear", "area"])
    b.add_argument("--max-per-class", type=int, default=None)
    b.add_argument("--workers", type=int, default=2)
    b.add_argument("--techniques", nargs="*", default=None)
    b.add_argument("--mode", default="resize", choices=["resize", "fixed"],
                   help="resize: original construction; fixed: first size*size*3 bytes, no resizing")
    r = sub.add_parser("run")
    r.add_argument("--work", required=True)
    r.add_argument("--seeds", type=int, nargs="+", default=[0])
    r.add_argument("--clfs", nargs="+", default=["cnn1"])
    r.add_argument("--dae-archs", nargs="+", default=["paper", "unet"],
                   choices=["paper", "unet"])
    r.add_argument("--dae-train", default="benign", choices=["benign", "both"])
    r.add_argument("--dae-epochs", type=int, default=20)
    r.add_argument("--clf-epochs", type=int, default=20)
    r.add_argument("--batch", type=int, default=64)
    r.add_argument("--patience", type=int, default=4)
    a = ap.parse_args()
    if a.cmd == "build":
        build_store(a.benign, a.malware, a.work, a.size, a.resample,
                    a.max_per_class, a.workers, a.techniques, a.mode)
    else:
        cmd_run(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/dae_study/report.py
"""
Aggregate results into tables.   python report.py --work OUT

Writes OUT/report/*.csv and OUT/report/report.md.  Mean +/- std over seeds,
paired deltas (same seed, same classifier, same technique), win counts.

Baselines (identity, augmentation, blur, blur_matched, oracle) are stored once
and joined into every DAE table (one table per DAE architecture / training set).
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

METRICS = ["auroc", "auprc", "tpr@fpr1e-2", "acc", "sens", "spec"]
COND_ORDER = ["clean_ref", "clean_subset", "oracle", "identity", "augmentation", "blur", "blur_matched",
              "dae_t", "dae_u", "dae_u_matched"]
# (defence, baseline) pairs worth reporting
PAIRS = [("blur", "identity"), ("blur_matched", "augmentation"),
         ("dae_t", "identity"), ("dae_t", "augmentation"), ("dae_t", "blur"),
         ("dae_u", "identity"), ("dae_u", "augmentation"), ("dae_u", "blur"),
         ("dae_u_matched", "augmentation"), ("dae_u_matched", "blur_matched")]
RCOLS = ["mse", "mse_identity", "psnr", "psnr_identity", "ssim", "ssim_identity"]


def fmt(m, s):
    return f"{m:.3f}±{0 if np.isnan(s) else s:.3f}"


def md_table(df):
    cols = [str(c) for c in df.columns]
    head = "| " + " | ".join([df.index.name or ""] + cols) + " |"
    sep = "|" + "---|" * (len(cols) + 1)
    rows = ["| " + " | ".join([str(i)] + [str(v) for v in r]) + " |"
            for i, r in zip(df.index, df.values)]
    return "\n".join([head, sep] + rows)


def read_all(work, pattern):
    files = sorted((work / "results").glob(pattern))
    return pd.concat([pd.read_json(p, lines=True) for p in files], ignore_index=True) \
        if files else pd.DataFrame()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work", required=True)
    a = ap.parse_args()
    work = Path(a.work)
    out = work / "report"
    out.mkdir(exist_ok=True)

    df = read_all(work, "raw_*.jsonl")
    if df.empty:
        raise SystemExit("no results yet")
    base = df[df.dae_train == "none"]
    dae = df[df.dae_train != "none"]
    groups = [(dt, ar, pd.concat([base, g], ignore_index=True))
              for (dt, ar), g in dae.groupby(["dae_train", "dae_arch"])] \
        or [("none", "none", base)]

    md = ["# Results\n"]
    deltas, aggs = [], []
    for dtrain, arch, full in groups:
        md.append(f"\n# DAE architecture `{arch}`, DAE trained on `{dtrain}`\n")
        aggs.append(full.assign(table=f"{arch}/{dtrain}"))
        for clf, g in full.groupby("clf"):
            md.append(f"\n## AUROC, classifier `{clf}`\n")
            s = g.groupby(["tech", "condition"])["auroc"].agg(["mean", "std", "count"])
            techs = list(dict.fromkeys(s.index.get_level_values(0)))
            conds = [c for c in COND_ORDER if c in s.index.get_level_values(1)]
            tab = pd.DataFrame({c: {t: fmt(s.loc[(t, c), "mean"], s.loc[(t, c), "std"])
                                    for t in techs if (t, c) in s.index}
                                for c in conds}).reindex(techs).fillna("—")
            tab.index.name = "technique"
            md.append(md_table(tab))
            md.append(f"\n(seeds per cell: {int(s['count'].min())}–{int(s['count'].max())})\n")

        piv = full.pivot_table(index=["clf", "tech", "seed"], columns="condition",
                               values="auroc").reset_index()
        rows = []
        for (clf, tech), g in piv.groupby(["clf", "tech"]):
            if tech == "clean":
                continue
            for cond, bs in PAIRS:
                if cond in g and bs in g:
                    d = (g[cond] - g[bs]).dropna()
                    rows.append({"dae_arch": arch, "dae_train": dtrain, "clf": clf,
                                 "tech": tech, "pair": f"{cond} vs {bs}",
                                 "n_seeds": len(d), "mean": d.mean(),
                                 "std": d.std(ddof=0), "wins": int((d > 0).sum())})
        if rows:
            dl = pd.DataFrame(rows)
            deltas.append(dl)
            md.append("\n## Paired ΔAUROC (positive = first method better; `wins` = seeds won)\n")
            for clf, g in dl.groupby("clf"):
                md.append(f"\n### `{clf}`\n")
                t = g.assign(v=[f"{m:+.3f}±{s:.3f} ({w}/{n})" for m, s, w, n in
                                zip(g["mean"], g["std"], g.wins, g.n_seeds)])
                t = t.pivot_table(index="tech", columns="pair", values="v", aggfunc="first")
                t = t[[f"{c} vs {b}" for c, b in PAIRS if f"{c} vs {b}" in t.columns]]
                t.index.name = "technique"
                md.append(md_table(t.fillna("—")))

    pd.concat(aggs).groupby(["table", "clf", "tech", "condition"])[METRICS].agg(
        ["mean", "std"]).to_csv(out / "classification_mean_std.csv")
    if deltas:
        pd.concat(deltas).to_csv(out / "paired_deltas.csv", index=False)

    rc = read_all(work, "recon_*.jsonl")
    if not rc.empty:
        rc.groupby(["dae_train", "dae_arch", "dae", "tech", "cls"])[RCOLS].mean().to_csv(
            out / "reconstruction.csv")
        md.append("\n# Reconstruction vs the clean image (all test files, mean over seeds)\n")
        md.append("`*_identity` = distance of the obfuscated image itself from clean. "
                  "A DAE should beat both that and the BLUR row; if it does not, it has "
                  "not learned to undo the obfuscation.\n")
        b = rc[rc.dae == "BLUR"]
        if not b.empty:
            t = b[b.cls == "all"].groupby("tech")[RCOLS].mean().round(4)
            t.index.name = "technique"
            md.append("\n### BLUR baseline (non-learned)\n")
            md.append(md_table(t))
        for (dt, ar, dn), g in rc[rc.dae != "BLUR"].groupby(["dae_train", "dae_arch", "dae"]):
            t = g[g.cls == "all"].groupby("tech")[RCOLS].mean().round(4)
            t.index.name = "technique"
            md.append(f"\n### DAE `{dn}`, architecture `{ar}`, trained on `{dt}`\n")
            md.append(md_table(t))

    (out / "report.md").write_text("\n".join(md))
    print(f"wrote {out / 'report.md'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/dae_study/extra.py
"""
Additional analyses added for the revision (reviewer requests).

Every sub-command takes --work, the folder of a finished run (the folder that
holds store.json, the image arrays, meta.csv and models/).  Nothing here
changes the main experiment; results go to <work>/results/.

  scores    save the per-file test score of every condition (uses the cached
            models; --allow-train retrains a missing one)
  stats     paired bootstrap over test files: confidence interval of the AUROC
            difference between two defences, and an equivalence check
  matched   size-controlled evaluation: AUROC inside strata of equal file size
            (and size x entropy), for the CNNs and for a metadata-only model
  neardup   how similar are the test images to the training images, and how do
            the results change when near-duplicates are removed
  loo       leave-one-obfuscation-out: train on all types except one, test on
            the held-out type (unseen transformation)
  sens      sensitivity of the DAE result to learning rate, width, loss,
            training length and training mixture (validation split only)

Typical order:  scores -> stats -> matched -> neardup ;  loo and sens train
new models (the slow part).
"""
import argparse
import itertools
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np

from data import Store, make_split
from metrics import clf_metrics
from models import Blur, batch_iter, build_cnn, build_dae, fit, predict_scores

MAIN_TECHS = ["Code_Packing", "Control_Flow_Obfuscation", "Data_Obfuscation",
              "Encryption", "String_Obfuscation"]          # the five invertible types
COMPS = [("dae_u_matched", "augmentation"), ("dae_t", "augmentation"),
         ("blur_matched", "augmentation"), ("dae_u_matched", "blur_matched")]
MARGINS = (0.01, 0.02, 0.03)


# ------------------------------------------------------------------ helpers
def ns(a, **kw):
    """Namespace with the training settings of the main run."""
    d = dict(work=a.work, dae_train="benign", dae_epochs=a.dae_epochs,
             clf_epochs=a.clf_epochs, batch=a.batch, patience=4)
    d.update(kw)
    return SimpleNamespace(**d)


def append(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")


def splits(labels, seed):
    sp = make_split(labels, seed)
    return tuple(np.where(sp == k)[0] for k in (0, 1, 2))


class WAuc:
    """AUROC with case weights and exact tie handling (fast: one sort per set)."""
    def __init__(self, s, y):
        self.y = np.asarray(y).astype(bool)
        u, self.inv = np.unique(np.asarray(s, np.float64), return_inverse=True)
        self.k = len(u)

    def __call__(self, w=None):
        w = np.ones(len(self.y)) if w is None else w
        wp = np.bincount(self.inv, weights=w * self.y, minlength=self.k)
        wn = np.bincount(self.inv, weights=w * ~self.y, minlength=self.k)
        den = wp.sum() * wn.sum()
        if den <= 0:
            return float("nan")
        cn = np.cumsum(wn) - wn
        return float((wp * (cn + 0.5 * wn)).sum() / den)


def clf_path(a, kind, seed, cache):
    return Path(a.work) / "models" / f"clf_{kind}_{cache}_seed{seed}.keras"


def dae_path(a, arch, seed, name):
    return Path(a.work) / "models" / f"dae_{arch}_{name}_benign_seed{seed}.keras"


def check_cached(a, path):
    if not path.exists() and not getattr(a, "allow_train", False):
        raise SystemExit(f"missing cached model {path.name}; copy the models/ folder of the "
                         f"main run into --work, or pass --allow-train to retrain it")


def load_scores(res, tag):
    f = Path(res) / tag
    return dict(np.load(f)) if f.exists() else None


# ------------------------------------------------------------------- scores
def cmd_scores(a):
    from run_experiment import get_clf, get_dae
    store = Store(a.work)
    labels, techs = store.labels, store.techs
    invertible = set(store.info["invertible"])
    res = Path(a.work) / "results"
    res.mkdir(exist_ok=True)
    blur = Blur(4)
    cfg = ns(a)
    for seed in a.seeds:
        tr, va, te = splits(labels, seed)
        dtr, dva = tr[labels[tr] == 0], va[labels[va] == 0]
        for kind in a.clfs:
            for arch in a.dae_archs:
                out = res / f"scores_{arch}_seed{seed}_{kind}.npz"
                if out.exists():
                    continue
                paths = [clf_path(a, kind, seed, c) for c in
                         ("clean", "aug", "blurmatched", f"daematched-{arch}-benign")]
                paths += [dae_path(a, arch, seed, n) for n in techs + ["UNION"]]
                for p in paths:
                    check_cached(a, p)
                clf_clean = get_clf(cfg, store, kind, seed, "clean", tr, va, ["clean"], None)
                clf_aug = get_clf(cfg, store, kind, seed, "aug", tr, va, ["clean"] + techs, None)
                clf_bm = get_clf(cfg, store, kind, seed, "blurmatched", tr, va, ["clean"] + techs, blur)
                d = {t: get_dae(cfg, store, seed, arch, t, [t], dtr, dva) for t in techs}
                du = get_dae(cfg, store, seed, arch, "UNION", ["clean"] + techs, dtr, dva)
                clf_dm = get_clf(cfg, store, kind, seed, f"daematched-{arch}-benign", tr, va,
                                 ["clean"] + techs, du)
                S = {}

                def put(cond, tech, idx, s):
                    S[f"{cond}|{tech}"] = np.asarray(s, np.float32)
                    S[f"idx|{tech}"] = np.asarray(idx)

                s_clean = predict_scores(clf_clean, store, te, "clean")
                put("clean_ref", "clean", te, s_clean)
                put("augmentation", "clean", te, predict_scores(clf_aug, store, te, "clean"))
                put("blur_matched", "clean", te, predict_scores(clf_bm, store, te, "clean", dae=blur))
                put("dae_u", "clean", te, predict_scores(clf_clean, store, te, "clean", dae=du))
                put("dae_u_matched", "clean", te, predict_scores(clf_dm, store, te, "clean", dae=du))
                for t in techs:
                    mask = store.valid[t][te]
                    idx = te[mask]
                    if len(idx) == 0:
                        continue
                    put("identity", t, idx, predict_scores(clf_clean, store, idx, t))
                    put("augmentation", t, idx, predict_scores(clf_aug, store, idx, t))
                    put("blur_matched", t, idx, predict_scores(clf_bm, store, idx, t, dae=blur))
                    put("dae_t", t, idx, predict_scores(clf_clean, store, idx, t, dae=d[t]))
                    put("dae_u", t, idx, predict_scores(clf_clean, store, idx, t, dae=du))
                    put("dae_u_matched", t, idx, predict_scores(clf_dm, store, idx, t, dae=du))
                    put("oracle" if t in invertible else "clean_subset", t, idx, s_clean[mask])
                np.savez_compressed(out, **S)
                print("saved", out.name, flush=True)


# -------------------------------------------------------------------- stats
def bootstrap_compare(sets, labels, splits_te, techs, comps, B, rng):
    """
    sets      : {seed: {key: array}}   (scores files)
    returns rows (list of dict).  The bootstrap resamples test files
    (stratified by class) with the SAME draw for every condition, so the
    paired structure is kept; the statistic is the mean over seeds of the
    AUROC difference.
    """
    seeds = sorted(sets)
    conds = sorted({c for ab in comps for c in ab})
    objs = {}
    for sd in seeds:
        te = splits_te[sd]
        for t in techs:
            for c in conds:
                k = f"{c}|{t}"
                if k in sets[sd]:
                    idx = sets[sd][f"idx|{t}"]
                    objs[(sd, t, c)] = (WAuc(sets[sd][k], labels[idx]), np.searchsorted(te, idx))
    point = {}
    boot = {}
    for t, (ca, cb) in itertools.product(techs, comps):
        if not all((sd, t, c) in objs for sd in seeds for c in (ca, cb)):
            continue
        d = np.array([objs[(sd, t, ca)][0]() - objs[(sd, t, cb)][0]() for sd in seeds])
        point[(t, ca, cb)] = d
        boot[(t, ca, cb)] = np.zeros((B, len(seeds)))
    for b in range(B):
        for si, sd in enumerate(seeds):
            te = splits_te[sd]
            w = np.zeros(len(te))
            for cls in (0, 1):
                pos = np.where(labels[te] == cls)[0]
                if len(pos):
                    w[pos] = rng.multinomial(len(pos), np.full(len(pos), 1.0 / len(pos)))
            for (t, ca, cb), arr in boot.items():
                (oa, pa), (ob, pb) = objs[(sd, t, ca)], objs[(sd, t, cb)]
                arr[b, si] = oa(w[pa]) - ob(w[pb])
    rows = []

    def summarise(name, d, bs, ca, cb):
        m = bs.mean(1)
        lo, hi = np.percentile(m, [2.5, 97.5])
        lo90, hi90 = np.percentile(m, [5, 95])
        r = {"tech": name, "comparison": f"{ca} - {cb}", "n_seeds": len(seeds),
             "mean_delta": float(d.mean()), "seed_sd": float(d.std(ddof=1)) if len(d) > 1 else float("nan"),
             "seeds_won": int((d > 0).sum()), "ci95_lo": float(lo), "ci95_hi": float(hi),
             "ci90_lo": float(lo90), "ci90_hi": float(hi90)}
        for mg in MARGINS:
            r[f"equiv_{mg}"] = bool(lo90 > -mg and hi90 < mg)
        rows.append(r)

    for (t, ca, cb), d in point.items():
        summarise(t, d, boot[(t, ca, cb)], ca, cb)
    for ca, cb in comps:
        ts = [t for t in MAIN_TECHS if (t, ca, cb) in point]
        if len(ts) >= 2:
            d = np.mean([point[(t, ca, cb)] for t in ts], 0)
            bs = np.mean([boot[(t, ca, cb)] for t in ts], 0)
            summarise(f"MEAN_{len(ts)}_types", d, bs, ca, cb)
    return rows


def write_csv(rows, path):
    import csv
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, keys)
        w.writeheader()
        w.writerows(rows)


def cmd_stats(a):
    store = Store(a.work)
    labels = store.labels
    res = Path(a.work) / "results"
    rng = np.random.default_rng(2026)
    allrows = []
    tes = {sd: splits(labels, sd)[2] for sd in a.seeds}
    for arch in a.dae_archs:
        for kind in a.clfs:
            if a.loo:
                sets = {}
                for sd in a.seeds:
                    S = {}
                    for f in sorted(res.glob(f"loo_*_seed{sd}_{kind}.npz")):
                        S.update(dict(np.load(f)))
                    if S:
                        sets[sd] = S
                comps = [("dae_u_matched_loo", "augmentation_loo"), ("dae_u_loo", "identity")]
                techs = sorted({k.split("|")[1] for s in sets.values() for k in s if not k.startswith("idx")})
            else:
                sets = {sd: load_scores(res, f"scores_{arch}_seed{sd}_{kind}.npz") for sd in a.seeds}
                sets = {k: v for k, v in sets.items() if v}
                comps, techs = COMPS, store.techs
            if len(sets) < 1:
                print("no scores for", arch, kind)
                continue
            rows = bootstrap_compare(sets, labels, tes, techs, comps, a.boot, rng)
            for r in rows:
                r.update(arch=arch, clf=kind)
            allrows += rows
            if a.loo:
                break
    name = "paired_bootstrap_loo.csv" if a.loo else "paired_bootstrap.csv"
    write_csv(allrows, res / name)
    print(f"wrote {res / name}  ({len(allrows)} rows; B={a.boot})")
    print("\nDAE (union, matched) minus augmentation; 95% CI over test files, mean of seeds")
    print("| arch | clf | transformation | mean | 95% CI | seeds won | equiv. +-0.02 |")
    print("|---|---|---|---|---|---|---|")
    for r in allrows:
        if r["comparison"].startswith(("dae_u_matched - augmentation", "dae_u_matched_loo")):
            print(f"| {r['arch']} | {r['clf']} | {r['tech']} | {r['mean_delta']:+.3f} | "
                  f"[{r['ci95_lo']:+.3f}, {r['ci95_hi']:+.3f}] | {r['seeds_won']}/{r['n_seeds']} | "
                  f"{'yes' if r['equiv_0.02'] else 'no'} |")


# ------------------------------------------------------------------ matched
def strata_ids(meta, variant, ref):
    """Integer stratum id per file from quantile bins of size (and entropy)."""
    ls = np.log10(meta["bytes"].to_numpy(float))
    en = meta["entropy"].to_numpy(float)
    ref_ls, ref_en = np.log10(ref["bytes"].to_numpy(float)), ref["entropy"].to_numpy(float)
    q = lambda v, r, k: np.clip(np.searchsorted(np.quantile(r, np.linspace(0, 1, k + 1)[1:-1]), v), 0, k - 1)
    if variant == "size10":
        return q(ls, ref_ls, 10)
    if variant == "size_entropy":
        return q(ls, ref_ls, 6) * 6 + q(en, ref_en, 6)
    raise ValueError(variant)


def stratified_auc(y, s, strata):
    """Pairs-weighted mean of within-stratum AUROCs (Mantel-Haenszel style)."""
    num = den = 0.0
    used = 0
    for g in np.unique(strata):
        m = strata == g
        npos, nneg = int(y[m].sum()), int((~y[m].astype(bool)).sum())
        if npos == 0 or nneg == 0:
            continue
        w = npos * nneg
        num += w * WAuc(s[m], y[m])()
        den += w
        used += int(m.sum())
    return (num / den if den else float("nan")), used


def cmd_matched(a):
    import pandas as pd
    from sklearn.ensemble import HistGradientBoostingClassifier
    store = Store(a.work)
    labels = store.labels
    res = Path(a.work) / "results"
    meta = pd.read_csv(Path(a.work) / "meta.csv")
    # --min-bytes N: keep only files with at least N bytes.  With the fixed
    # byte budget (N = size*size*3) these files fill the whole image, so no
    # padding can reveal their length.
    big = meta["bytes"].to_numpy() >= a.min_bytes
    rows = []
    conds = ["clean_ref", "identity", "augmentation", "dae_t", "dae_u_matched", "oracle", "clean_subset"]
    for seed in a.seeds:
        tr, va, te = splits(labels, seed)
        te = te[big[te]]
        meta_models = {}
        for name, cols in (("meta_size", ["bytes"]), ("meta_size_entropy", ["bytes", "entropy"])):
            m = HistGradientBoostingClassifier(random_state=0).fit(meta.loc[tr, cols], labels[tr])
            meta_models[name] = (cols, m)
        for variant in ("none", "size10", "size_entropy"):
            st = None if variant == "none" else strata_ids(meta, variant, meta.loc[tr])
            for name, (cols, m) in meta_models.items():
                s = m.predict_proba(meta.loc[te, cols])[:, 1]
                y = labels[te]
                auc, used = (WAuc(s, y)(), len(te)) if st is None else stratified_auc(y, s, st[te])
                rows.append(dict(seed=seed, arch="-", clf="-", tech="clean", cond=name,
                                 variant=variant, auroc=auc, n_used=used))
            for arch in a.dae_archs:
                for kind in a.clfs:
                    S = load_scores(res, f"scores_{arch}_seed{seed}_{kind}.npz")
                    if S is None:
                        continue
                    for key in S:
                        if key.startswith("idx|"):
                            continue
                        cond, tech = key.split("|")
                        if cond not in conds:
                            continue
                        idx = S[f"idx|{tech}"]
                        keep = big[idx]
                        idx, sc = idx[keep], S[key][keep]
                        y = labels[idx]
                        auc, used = (WAuc(sc, y)(), len(idx)) if st is None else \
                            stratified_auc(y, sc, st[idx])
                        rows.append(dict(seed=seed, arch=arch, clf=kind, tech=tech, cond=cond,
                                         variant=variant, auroc=auc, n_used=used))
    write_csv(rows, res / "matched_eval.csv")
    df = pd.DataFrame(rows)
    print("mean over seeds; 'none' = ordinary AUROC, size10 / size_entropy = inside strata of equal size (and entropy)")
    t = df[(df.tech == "clean") & df.cond.isin(["clean_ref", "meta_size", "meta_size_entropy"])]
    print(t.pivot_table(index=["arch", "clf", "cond"], columns="variant", values="auroc").round(3).to_string())
    print("\nobfuscated types, U-Net, strata of size x entropy:")
    t = df[(df.arch == "unet") & (df.variant == "size_entropy") & df.cond.isin(["identity", "augmentation", "dae_u_matched"])]
    print(t.pivot_table(index=["clf", "tech"], columns="cond", values="auroc").round(3).to_string())
    print("wrote", res / "matched_eval.csv")


# ----------------------------------------------------------------- neardup
def pooled_vectors(store, k=16, chunk=500):
    n = store.info["n"]
    out = np.zeros((n, k * k * 3), np.float32)
    f = store.size // k
    for s in range(0, n, chunk):
        x = store.arrays["clean"][s:s + chunk].astype(np.float32) / 255.0
        x = x.reshape(len(x), k, f, k, f, 3).mean((2, 4))
        out[s:s + chunk] = x.reshape(len(x), -1)
    out -= out.mean(0, keepdims=True)
    out /= np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-8)
    return out


def cmd_neardup(a):
    import pandas as pd
    store = Store(a.work)
    labels = store.labels
    res = Path(a.work) / "results"
    V = pooled_vectors(store)
    rows = []
    for seed in a.seeds:
        tr, va, te = splits(labels, seed)
        sim = V[te] @ V[tr].T
        j = sim.argmax(1)
        smax = sim[np.arange(len(te)), j]
        nn_label = labels[tr][j]
        np.savez_compressed(res / f"neardup_seed{seed}.npz", te=te, smax=smax, nn_label=nn_label)
        for thr in a.thr:
            near = smax >= thr
            rows.append(dict(seed=seed, thr=thr, share_near=float(near.mean()),
                             nn_label_agrees_near=float((nn_label[near] == labels[te][near]).mean()) if near.any() else float("nan"),
                             nn_label_agrees_far=float((nn_label[~near] == labels[te][~near]).mean()) if (~near).any() else float("nan")))
        for arch in a.dae_archs:
            for kind in a.clfs:
                S = load_scores(res, f"scores_{arch}_seed{seed}_{kind}.npz")
                if S is None:
                    continue
                for thr in a.thr:
                    keep = smax < thr                      # drop test files with a near-duplicate in train
                    kept_files = te[keep]
                    for cond in ("clean_ref", "augmentation", "dae_u_matched"):
                        s_all = S[f"{cond}|clean"]
                        y = labels[te]
                        rows.append(dict(seed=seed, arch=arch, clf=kind, thr=thr, cond=cond,
                                         n_kept=int(keep.sum()),
                                         auroc_all=WAuc(s_all, y)(),
                                         auroc_dedup=WAuc(s_all[keep], y[keep])() if keep.sum() > 10 else float("nan")))
    write_csv(rows, res / "neardup.csv")
    df = pd.DataFrame(rows)
    print(df[df.cond.isna()].groupby("thr")[["share_near", "nn_label_agrees_near", "nn_label_agrees_far"]].mean().round(3).to_string())
    if "cond" in df and df.cond.notna().any():
        print(df[df.cond.notna()].groupby(["arch", "clf", "cond", "thr"])[["n_kept", "auroc_all", "auroc_dedup"]].mean().round(3).to_string())
    print("wrote", res / "neardup.csv")


# --------------------------------------------------------------------- loo
def cmd_loo(a):
    from run_experiment import get_clf, get_dae
    store = Store(a.work)
    labels, techs = store.labels, store.techs
    res = Path(a.work) / "results"
    res.mkdir(exist_ok=True)
    cfg = ns(a)
    held = a.held or [t for t in MAIN_TECHS if t in techs]
    raw = res / "raw_loo.jsonl"
    for seed in a.seeds:
        tr, va, te = splits(labels, seed)
        dtr, dva = tr[labels[tr] == 0], va[labels[va] == 0]
        for h in held:
            mix = ["clean"] + [t for t in techs if t != h]
            du = None
            for kind in a.clfs:
                out = res / f"loo_{h}_seed{seed}_{kind}.npz"
                if out.exists():
                    continue
                if du is None:
                    du = get_dae(cfg, store, seed, "unet", f"LOO-{h}", mix, dtr, dva)
                clf_clean = get_clf(cfg, store, kind, seed, "clean", tr, va, ["clean"], None)
                clf_aug = get_clf(cfg, store, kind, seed, f"loo-{h}-aug", tr, va, mix, None)
                clf_dm = get_clf(cfg, store, kind, seed, f"loo-{h}-daematched", tr, va, mix, du)
                idx = te[store.valid[h][te]]
                S = {f"idx|{h}": idx}
                for cond, (clf, dae) in {"identity": (clf_clean, None),
                                         "augmentation_loo": (clf_aug, None),
                                         "dae_u_loo": (clf_clean, du),
                                         "dae_u_matched_loo": (clf_dm, du)}.items():
                    s = predict_scores(clf, store, idx, h, dae=dae)
                    S[f"{cond}|{h}"] = s.astype(np.float32)
                    append(raw, {"seed": seed, "clf": kind, "held_out": h, "condition": cond,
                                 **clf_metrics(labels[idx], s)})
                np.savez_compressed(out, **S)
                print(f"seed {seed} / {kind} / held out {h} done", flush=True)


# -------------------------------------------------------------------- sens
VARIANTS = {
    "default": {},
    "lr_3e-4": {"lr": 3e-4},
    "lr_3e-3": {"lr": 3e-3},
    "width_0.5": {"width": 0.5},
    "width_2": {"width": 2.0},
    "loss_mae": {"loss": "mae"},
    "epochs_5": {"epochs": 5},
    "epochs_40": {"epochs": 40, "patience": 8},
    "mix_clean_0.5": {"p_clean": 0.5},
}


def cmd_sens(a):
    store = Store(a.work)
    labels, techs = store.labels, store.techs
    res = Path(a.work) / "results"
    res.mkdir(exist_ok=True)
    mdir = Path(a.work) / "models_sens"
    mdir.mkdir(exist_ok=True)
    out_csv = res / "sens_raw.jsonl"
    names = list(VARIANTS) if a.variants == ["all"] else a.variants
    kind = a.clfs[0]
    for seed in a.seeds:
        tr, va, te = splits(labels, seed)
        dtr, dva = tr[labels[tr] == 0], va[labels[va] == 0]
        mix = ["clean"] + techs
        for name in names:
            v = VARIANTS[name]
            done = res / f"sens_{name}_seed{seed}.flag"
            if done.exists():
                continue
            pc = v.get("p_clean")
            ep, pat = v.get("epochs", a.dae_epochs), v.get("patience", 4)
            import tensorflow as tf

            def get(path, build, train_it, val_it, epochs, patience, tag):
                if path.exists():
                    return tf.keras.models.load_model(path)
                tf.keras.utils.set_random_seed(seed)
                m = build()
                fit(m, train_it, val_it, epochs, patience, tag=tag)
                m.save(path)
                return m

            rng = np.random.default_rng(seed)
            du = get(mdir / f"dae_{name}_seed{seed}.keras",
                     lambda: build_dae(store.size, "unet", v.get("width", 1.0), v.get("lr"), v.get("loss", "mse")),
                     lambda: batch_iter(store, dtr, mix, a.batch, rng, target="clean", p_clean=pc),
                     lambda: batch_iter(store, dva, mix, a.batch, np.random.default_rng(999),
                                        target="clean", shuffle=False, p_clean=pc),
                     ep, pat, f"sens-DAE[{name}] seed{seed}")
            rng2 = np.random.default_rng(seed)
            clf_dm = get(mdir / f"clf_{kind}_dm_{name}_seed{seed}.keras",
                         lambda: build_cnn(kind, store.size),
                         lambda: batch_iter(store, tr, mix, a.batch, rng2, dae=du, p_clean=pc),
                         lambda: batch_iter(store, va, mix, a.batch, np.random.default_rng(999),
                                            dae=du, shuffle=False, p_clean=pc),
                         a.clf_epochs, 4, f"sens-{kind}-matched[{name}] seed{seed}")
            rng3 = np.random.default_rng(seed)
            clf_aug = get(mdir / f"clf_{kind}_aug_{name if pc is not None else 'default'}_seed{seed}.keras",
                          lambda: build_cnn(kind, store.size),
                          lambda: batch_iter(store, tr, mix, a.batch, rng3, p_clean=pc),
                          lambda: batch_iter(store, va, mix, a.batch, np.random.default_rng(999),
                                             shuffle=False, p_clean=pc),
                          a.clf_epochs, 4, f"sens-{kind}-aug[{name}] seed{seed}")
            for t in techs:           # evaluation on the VALIDATION split only
                idx = va[store.valid[t][va]]
                s_dm = predict_scores(clf_dm, store, idx, t, dae=du)
                s_aug = predict_scores(clf_aug, store, idx, t)
                m_dm, m_aug = clf_metrics(labels[idx], s_dm), clf_metrics(labels[idx], s_aug)
                append(out_csv, {"variant": name, "seed": seed, "clf": kind, "tech": t,
                                 "auroc_dae_matched": m_dm["auroc"], "auroc_augmentation": m_aug["auroc"],
                                 "delta": m_dm["auroc"] - m_aug["auroc"]})
            done.write_text("ok")
            print(f"sens {name} seed {seed} done", flush=True)
    summarize_sens(res)


def summarize_sens(res):
    import pandas as pd
    f = Path(res) / "sens_raw.jsonl"
    if not f.exists():
        return
    df = pd.read_json(f, lines=True)
    main = df[df.tech.isin(MAIN_TECHS)]
    t = main.groupby("variant")[["auroc_dae_matched", "auroc_augmentation", "delta"]].mean().round(3)
    t["min_delta"] = main.groupby(["variant", "tech"])["delta"].mean().groupby("variant").min().round(3)
    t["max_delta"] = main.groupby(["variant", "tech"])["delta"].mean().groupby("variant").max().round(3)
    print("validation split; mean over the five invertible types and seeds")
    print(t.to_string())
    t.to_csv(Path(res) / "sens_summary.csv")


# ---------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    def common(p):
        p.add_argument("--work", required=True)
        p.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
        p.add_argument("--clfs", nargs="+", default=["cnn1", "cnn2"])
        p.add_argument("--dae-archs", nargs="+", default=["unet"], choices=["paper", "unet"])
        p.add_argument("--batch", type=int, default=64)
        p.add_argument("--dae-epochs", type=int, default=20)
        p.add_argument("--clf-epochs", type=int, default=20)
        p.add_argument("--allow-train", action="store_true")
        return p
    common(sub.add_parser("scores"))
    p = common(sub.add_parser("stats"))
    p.add_argument("--boot", type=int, default=1000)
    p.add_argument("--loo", action="store_true", help="use the leave-one-out score files")
    p = common(sub.add_parser("matched"))
    p.add_argument("--min-bytes", type=int, default=0, help="keep only files of at least this size")
    p = common(sub.add_parser("neardup"))
    p.add_argument("--thr", type=float, nargs="+", default=[0.90, 0.95, 0.99])
    p = common(sub.add_parser("loo"))
    p.add_argument("--held", nargs="*", default=None)
    p = common(sub.add_parser("sens"))
    p.add_argument("--variants", nargs="+", default=["all"], choices=["all"] + list(VARIANTS))
    a = ap.parse_args()
    {"scores": cmd_scores, "stats": cmd_stats, "matched": cmd_matched, "neardup": cmd_neardup,
     "loo": cmd_loo, "sens": cmd_sens}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/dae_study/upx_counts.py
"""
How many files did UPX accept, per class?  (Reviewer request: report the
number and class distribution of packed files, the acceptance rates, and check
whether the refused files differ from the accepted ones.)

  python upx_counts.py --meta meta.csv --valid valid_UPX.npy [--upx-version "upx 4.2.2"]

meta.csv is written by the build step; valid_UPX.npy by the UPX run (a
boolean array, True where UPX produced a packed file).  Both are in the
results folder of the UPX run (dae_study_results_upx on Drive).
"""
import argparse

import numpy as np
import pandas as pd

from data import make_split

ap = argparse.ArgumentParser()
ap.add_argument("--meta", required=True)
ap.add_argument("--valid", required=True)
ap.add_argument("--upx-version", default="")
ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
a = ap.parse_args()

m = pd.read_csv(a.meta)
v = np.load(a.valid).astype(bool)
assert len(m) == len(v), (len(m), len(v))
m["accepted"] = v
m["class"] = m["label"].map({0: "benign", 1: "malware"})

print(f"UPX version: {a.upx_version or '(not given)'}\n")
print("| Class | Files | Packed by UPX | Acceptance rate |")
print("|---|---|---|---|")
for c, g in m.groupby("class"):
    print(f"| {c} | {len(g)} | {int(g.accepted.sum())} | {100 * g.accepted.mean():.1f}% |")
print(f"| all | {len(m)} | {int(v.sum())} | {100 * v.mean():.1f}% |")

acc = m[m.accepted]
print(f"\nShare of malware among packed files: {100 * acc.label.mean():.1f}% "
      f"(whole set: {100 * m.label.mean():.1f}%)")

print("\nAccepted against refused files (median):")
print("| Class | Group | n | Size (bytes) | Entropy |")
print("|---|---|---|---|---|")
for c in ("benign", "malware"):
    for name, g in (("accepted", m[(m["class"] == c) & m.accepted]),
                    ("refused", m[(m["class"] == c) & ~m.accepted])):
        if len(g):
            print(f"| {c} | {name} | {len(g)} | {int(g.bytes.median()):,} | {g.entropy.median():.2f} |")

print("\nPacked files in the test split of each seed:")
print("| Seed | Test files | Packed benign | Packed malware |")
print("|---|---|---|---|")
labels = m.label.to_numpy()
for sd in a.seeds:
    te = np.where(make_split(labels, sd) == 2)[0]
    k = te[v[te]]
    print(f"| {sd} | {len(te)} | {int((labels[k] == 0).sum())} | {int((labels[k] == 1).sum())} |")

## Step 0 - Build the image store of the main study again (images are not kept between sessions)

In [ ]:
!python /content/dae_study/run_experiment.py build \
    --benign /content/trapmine/benign --malware /content/trapmine/malware \
    --work {WORK} --max-per-class {MAX_PER_CLASS} --workers 2

# new results go to Drive; the models of the main study are copied in so they are reused
import os, glob, shutil
for d in ("models", "results", "figures"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
    link = f"{WORK}/{d}"
    if not os.path.exists(link):
        os.symlink(f"{OUT}/{d}", link)
if not REHEARSAL:
    n = 0
    for f in glob.glob(f"{OUT_MAIN}/models/*.keras"):
        dst = f"{OUT}/models/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy2(f, dst); n += 1
    print(f"copied {n} cached models of the main study")
    # the image store must be the same as in the main study: compare the file list and labels
    import numpy as np, json
    old = json.load(open(f"{OUT_MAIN}/store.json"))
    new = json.load(open(f"{WORK}/store.json"))
    print("main study:", old["n"], "files;  rebuilt:", new["n"], "files")
    assert old["n"] == new["n"], "different number of files: the cached models would not match"
    import pandas as pd
    a_ = pd.read_csv(f"{OUT_MAIN}/meta.csv"); b_ = pd.read_csv(f"{WORK}/meta.csv")
    assert (a_["sha256"] == b_["sha256"]).all(), "different file order: the cached models would not match"
    print("same files, same order: OK")

## Part A - Statistics, size control, near-duplicates (no training)

In [ ]:
if DO_A_STATS:
    !python /content/dae_study/extra.py scores  --work {WORK} --seeds {SEEDS} --clfs {CLFS} {ALLOW} --dae-epochs {EPOCHS} --clf-epochs {EPOCHS}
    !python /content/dae_study/extra.py stats   --work {WORK} --seeds {SEEDS} --clfs {CLFS} --boot {BOOT}

In [ ]:
if DO_A_STATS:
    !python /content/dae_study/extra.py matched --work {WORK} --seeds {SEEDS} --clfs {CLFS}
    !python /content/dae_study/extra.py neardup --work {WORK} --seeds {SEEDS} --clfs {CLFS}

## Part B - UPX: how many files were packed?

In [ ]:
if DO_B_UPX:
    if not shutil.which("upx"):
        subprocess.run(["apt-get", "-qq", "install", "-y", "upx-ucl"], check=True)
    print("UPX of the current Colab image:", subprocess.run(["upx", "-V"], capture_output=True, text=True).stdout.splitlines()[0])
    print("NOTE: this is the UPX of the CURRENT Colab image. Check the first lines of the output of your UPX run "
          "(it printed 'upx -V'); if they differ, use the older one in the paper.")
    if REHEARSAL:
        print("(skipped in the rehearsal: needs the full UPX run files)")
    else:
        !python /content/dae_study/upx_counts.py --meta {OUT_UPX}/meta.csv --valid {OUT_UPX}/valid_UPX.npy

In [ ]:
if DO_B_UPX_STATS and not REHEARSAL:
    if not shutil.which("upx"):
        subprocess.run(["apt-get", "-qq", "install", "-y", "upx-ucl"], check=True)
    WORKU = "/content/work_rev_upx"
    OUTU  = OUT + "/upx"
    !python /content/dae_study/run_experiment.py build \
        --benign /content/trapmine/benign --malware /content/trapmine/malware \
        --work {WORKU} --max-per-class {MAX_PER_CLASS} --workers 2 --techniques UPX
    for d in ("models", "results", "figures"):
        os.makedirs(f"{OUTU}/{d}", exist_ok=True)
        if not os.path.exists(f"{WORKU}/{d}"):
            os.symlink(f"{OUTU}/{d}", f"{WORKU}/{d}")
    for f in glob.glob(f"{OUT_UPX}/models/*.keras"):
        dst = f"{OUTU}/models/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy2(f, dst)
    import numpy as np
    same = (np.load(f"{OUT_UPX}/valid_UPX.npy") == np.load(f"{WORKU}/valid_UPX.npy")).all()
    print("UPX accepted the same files as in the UPX run:", same)
    assert same, "different files were packed: the cached UPX models would not match"
    !python /content/dae_study/extra.py scores --work {WORKU} --seeds {SEEDS} --clfs {CLFS}
    !python /content/dae_study/extra.py stats  --work {WORKU} --seeds {SEEDS} --clfs {CLFS} --boot {BOOT}

## Part C - Unseen obfuscation: leave one type out (trains new models)

In [ ]:
if DO_C_LOO:
    !python /content/dae_study/extra.py loo   --work {WORK} --seeds {SEEDS} --clfs {CLFS} --held {LOO_HELD} --dae-epochs {EPOCHS} --clf-epochs {EPOCHS} {ALLOW}
    !python /content/dae_study/extra.py stats --work {WORK} --seeds {SEEDS} --clfs {CLFS} --boot {BOOT} --loo

## Part D - Sensitivity of the DAE result (trains new models; evaluated on the validation split only)

In [ ]:
if DO_D_SENS:
    !python /content/dae_study/extra.py sens --work {WORK} --seeds {SENS_SEEDS} --clfs {SENS_CLFS} --variants {SENS_VARIANTS} --dae-epochs {EPOCHS} --clf-epochs {EPOCHS}

## Part E - Fixed byte budget: no resizing, image side independent of file length (builds a second store and runs a smaller study)

In [ ]:
if DO_E_FIXED:
    WORKF = f"/content/work_rev_fixed_{MODE}"
    OUTF  = OUT + "/fixed_budget"
    !python /content/dae_study/run_experiment.py build \
        --benign /content/trapmine/benign --malware /content/trapmine/malware \
        --work {WORKF} --max-per-class {MAX_PER_CLASS} --workers 2 --mode fixed --techniques {FIXED_TECHS}
    for d in ("models", "results", "figures"):
        os.makedirs(f"{OUTF}/{d}", exist_ok=True)
        if not os.path.exists(f"{WORKF}/{d}"):
            os.symlink(f"{OUTF}/{d}", f"{WORKF}/{d}")
    !python /content/dae_study/run_experiment.py run --work {WORKF} --seeds {SEEDS} --clfs {CLFS} \
        --dae-archs unet --dae-epochs {EPOCHS} --clf-epochs {EPOCHS} --batch 64
    !python /content/dae_study/extra.py scores  --work {WORKF} --seeds {SEEDS} --clfs {CLFS} --dae-epochs {EPOCHS} --clf-epochs {EPOCHS}
    !python /content/dae_study/extra.py stats   --work {WORKF} --seeds {SEEDS} --clfs {CLFS} --boot {BOOT}
    # files that fill the whole image (>= 128*128*3 = 49,152 bytes): no padding can reveal their length
    !python /content/dae_study/extra.py matched --work {WORKF} --seeds {SEEDS} --clfs {CLFS} --min-bytes 49152
    !python /content/dae_study/report.py --work {WORKF}
    print(open(f"{WORKF}/report/report.md").read()[:6000])

## Save the small result files next to the big ones

In [ ]:
import glob
os.makedirs(f"{OUT}/summary", exist_ok=True)
for f in glob.glob(f"{OUT}/results/*.csv") + glob.glob(f"{OUT}/results/*.jsonl") + glob.glob(f"{OUT}/fixed_budget/results/*.csv"):
    shutil.copy(f, f"{OUT}/summary/" + os.path.basename(os.path.dirname(os.path.dirname(f))) + "_" + os.path.basename(f))
print(sorted(os.listdir(f"{OUT}/summary")))
print("All results are in Google Drive:", OUT)